In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers

%load_ext autoreload
%autoreload 2

from utils import scale_img

SEED = 143


I0000 00:00:1789313754.573111   13185 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789313759.560827   13185 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789313771.866673   13185 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Loading Data

In [3]:
df_train = pd.read_csv('Dataset/train.csv')
df_test = pd.read_csv('Dataset/test.csv')

In [4]:
x = df_train.drop(['label'], axis=1).values
y = df_train['label'].values

In [5]:
x = x.reshape(-1, 28, 28, 1)
x_test = df_test.values.reshape(-1, 28, 28, 1)

In [6]:
x_train, x_cv, y_train, y_cv = train_test_split(x, y, test_size=0.2, random_state=SEED)

print(x_train.shape, y_train.shape)


(33600, 28, 28, 1) (33600,)


In [7]:
x_train_scaled, x_cv_scaled, x_test_scaled = scale_img(x_train, x_cv, x_test)

# Building the model

In [8]:
model = tf.keras.Sequential([
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(28, 28 ,1)),
    
    layers.MaxPool2D(2,2),
    
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    
    layers.MaxPool2D(2,2),
    
    layers.Flatten(),
    
    layers.Dense(128, activation='relu'),
    
    layers.Dense(10, activation='linear'),
])

/home/kiamehr/.pyenv/versions/3.12.10/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789313808.746229   13185 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789313808.810863   21873 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789313808.904490   13185 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly

In [11]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(0.001),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(  
    monitor='val_loss',
    patience=3,
)

history = model.fit(
    x_train_scaled, y_train,
    validation_data=(x_cv_scaled, y_cv),
    callbacks=[early_stopping],
    epochs=10
)

W0000 00:00:1789313841.379018   13185 cpu_allocator_impl.cc:82] Allocation of 105369600 exceeds 10% of free system memory.


Epoch 1/10
   4/1050 ━━━━━━━━━━━━━━━━━━━━ 58s 56ms/step - accuracy: 0.1562 - loss: 2.2421

W0000 00:00:1789313846.523743   21882 cpu_allocator_impl.cc:82] Allocation of 20321280 exceeds 10% of free system memory.
W0000 00:00:1789313846.526023   21880 cpu_allocator_impl.cc:82] Allocation of 20321280 exceeds 10% of free system memory.
W0000 00:00:1789313846.581519   21879 cpu_allocator_impl.cc:82] Allocation of 20321280 exceeds 10% of free system memory.
W0000 00:00:1789313846.581674   21880 cpu_allocator_impl.cc:82] Allocation of 20321280 exceeds 10% of free system memory.


1050/1050 ━━━━━━━━━━━━━━━━━━━━ 50s 43ms/step - accuracy: 0.9467 - loss: 0.1707 - val_accuracy: 0.9792 - val_loss: 0.0665
Epoch 2/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 79s 40ms/step - accuracy: 0.9832 - loss: 0.0518 - val_accuracy: 0.9850 - val_loss: 0.0481
Epoch 3/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 49s 47ms/step - accuracy: 0.9883 - loss: 0.0352 - val_accuracy: 0.9836 - val_loss: 0.0514
Epoch 4/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 43s 41ms/step - accuracy: 0.9921 - loss: 0.0263 - val_accuracy: 0.9863 - val_loss: 0.0483
Epoch 5/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 43s 41ms/step - accuracy: 0.9940 - loss: 0.0181 - val_accuracy: 0.9889 - val_loss: 0.0437
Epoch 6/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 45s 43ms/step - accuracy: 0.9945 - loss: 0.0160 - val_accuracy: 0.9894 - val_loss: 0.0402
Epoch 7/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 50s 48ms/step - accuracy: 0.9962 - loss: 0.0117 - val_accuracy: 0.9883 - val_loss: 0.0444
Epoch 8/10
1050/1050 ━━━━━━━━━━━━━━━━━━━━ 80s 46ms/step - accuracy: 0.9970 - loss: 0.00

In [12]:
model.save('models/pro1.keras')